# 🌊 Flood Detection from Images — All-in-One Google Colab

**Upload satellite / drone / street-level images → automatic Flood vs. No-Flood detection with
full statistics, graphs and a downloadable analysis report (.zip).**

This notebook is **fully automatic**: it sets itself up, downloads the pre-trained model,
asks you for images, runs detection and exports every result. Just run the cells top-to-bottom
(`Runtime ▸ Run all`). Training a fresh model from scratch and satellite fine-tuning are
included as optional sections.

**What you get**

| # | Section | What happens |
|---|---------|--------------|
| 1 | Setup | GPU check + all dependencies + reproducibility seeds |
| 2 | Pre-trained model | Auto-downloads the fine-tuned flood model from Google Drive |
| 3 | Upload images | Pick single/multiple images **or** upload a whole `.zip` **or** run the demo |
| 4 | Detection | Flood / No-Flood verdict + probability for every image |
| 5 | Visual results | Annotated image gallery with verdicts & confidence |
| 6 | Stats & graphs | Summary table, pie chart, confidence histogram, top-N ranking, queries |
| 7 | Export | `results.csv` + renamed copies (`FLOOD_1.jpg` / `NO_FLOOD_1.jpg`) + report figure + all zipped & downloaded |
| 8 | Retrain (optional) | Full training pipeline on the flood dataset + training curves + confusion matrix + TFLite export |
| 9 | Satellite fine-tune (optional) | Adapt the model to satellite / drone imagery (your own dataset or FloodNet-style data) |

> ⚠️ **Satellite-image note:** the pre-trained model was trained mostly on *street-level* flood
> photos, so treat satellite/drone results as an estimate. Water is usually still detected well,
> but for production use, fine-tune on satellite imagery (Section 8/9 shows the pipeline).

> 💡 **GPU recommended but not required:** `Runtime ▸ Change runtime type ▸ T4 GPU`.
> Everything also runs on CPU (a bit slower).

## 1️⃣ Setup & Environment

Installs dependencies, checks for a GPU, imports everything and fixes random seeds so results
are reproducible.

In [ ]:
# ============================================================================
# 1. SETUP — dependencies, GPU check, imports, seeds
# ============================================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('🔧 Installing dependencies...')
pip_install('gdown')

import os, shutil, zipfile, random, warnings, datetime, textwrap
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from IPython.display import display   # rich display (tables, DataFrames)
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

LABELS    = ['Flooding', 'No Flooding']
IMG_SIZE  = (224, 224)
BATCH     = 32

gpu = tf.config.list_physical_devices('GPU')
print()
print('✅ Environment ready')
print('   TensorFlow :', tf.__version__)
print('   GPU        :', gpu[0].name if gpu else 'not available (CPU mode — still works, slightly slower)')

## 2️⃣ Load the Pre-Trained Flood Model *(automatic)*

Downloads the fine-tuned model (~28 MB) that ships with the GitHub repo. Because it was saved
in 2020 with old Keras, we rebuild the identical architecture in modern Keras 3 and transfer the
weights layer-by-layer — verified to reproduce the original predictions.

If the download ever fails (e.g. Drive quota), the notebook **automatically trains a fresh
model** from the flood dataset instead, so you are never blocked.

In [ ]:
# ============================================================================
# 2. PRE-TRAINED MODEL — download, rebuild, transfer weights (or auto-train)
# ============================================================================
DRIVE_MODEL_ID = '1mF8NmMClUbKXYJoDdW3OYHhwiSknI5hk'      # fine-tuned flood model
DRIVE_DATA_ID  = '1oT07-wVuYj_hByn4FqUtIj63Yg_fe7dP'      # dataset zip (train/valid/test)
MODELS_DIR = 'models'; os.makedirs(MODELS_DIR, exist_ok=True)
H5_PATH = os.path.join(MODELS_DIR, 'pretrained_flood_model.h5')


def build_flood_model(head_activation):
    '''MobileNet trunk + GAP + Dropout + Dense(2) head.'''
    mobile = tf.keras.applications.MobileNet(weights='imagenet', include_top=False)
    x = mobile.get_layer('conv_pw_13_relu').output          # last ReLU map (1024 ch)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling2d')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout')(x)
    # head MUST be named 'dense' to match the 2020 checkpoint for weight transfer
    out = tf.keras.layers.Dense(2, activation=head_activation, name='dense')(x)
    return tf.keras.models.Model(mobile.input, out, name='flood_net')


def get_dataset_dirs():
    '''Download + extract the flood dataset once; returns dict of split folders.'''
    zip_path = 'flood_dataset.zip'
    if not (os.path.isdir('data/train') or os.path.isdir('data/data/train')):
        import gdown
        print('⬇️  Downloading flood dataset (~52 MB)...')
        gdown.download(id=DRIVE_DATA_ID, output=zip_path, quiet=False)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('.')
    base = 'data' if os.path.isdir('data/train') else 'data/data'
    return {s: os.path.join(base, s) for s in ['train', 'valid', 'test']}


def train_fresh_model(epochs=10, verbose=2):
    '''Full automatic training pipeline (used as fallback + in Section 8).'''
    dirs = get_dataset_dirs()

    def make_ds(split, shuffle):
        ds = tf.keras.utils.image_dataset_from_directory(
            dirs[split], labels='inferred', label_mode='categorical',
            class_names=['flooding', 'normal'], image_size=IMG_SIZE,
            batch_size=BATCH, shuffle=shuffle, seed=SEED)
        return ds.map(lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
                      num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    train_ds, val_ds, test_ds = (make_ds('train', True), make_ds('valid', False),
                                 make_ds('test', False))

    model = build_flood_model('softmax')                    # valid probabilities
    for layer in model.layers[:-23]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])

    print(f'🚀 Training ({epochs} epochs max, early-stopping on, '
          f'{sum(1 for l in model.layers if l.trainable)}/{len(model.layers)} layers trainable)...')
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose,
                     callbacks=[tf.keras.callbacks.EarlyStopping(
                         monitor='val_accuracy', patience=3, restore_best_weights=True)])

    # evaluation
    y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in test_ds])
    y_pred = np.argmax(model.predict(test_ds, verbose=0), 1)
    print('\n📊 Fresh model test-set performance')
    print(f'   Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'   Precision: {precision_score(y_true, y_pred, average="macro"):.4f}')
    print(f'   Recall   : {recall_score(y_true, y_pred, average="macro"):.4f}')
    print(f'   F1 Score : {f1_score(y_true, y_pred, average="macro"):.4f}')
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=4))

    # training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(hist.history['accuracy'], 'o-', label='train')
    axes[0].plot(hist.history['val_accuracy'], 's-', label='validation')
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(hist.history['loss'], 'o-', label='train')
    axes[1].plot(hist.history['val_loss'], 's-', label='validation')
    axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=.3)
    plt.show()

    model.save(os.path.join(MODELS_DIR, 'flood_model_trained.keras'))
    return model, hist


# --- try the pre-trained route ------------------------------------------------
LEGACY_SIGMOID = False
model, model_source = None, None
try:
    if not os.path.exists(H5_PATH):
        import gdown
        print('⬇️  Downloading pre-trained model (~28 MB) from Google Drive...')
        gdown.download(id=DRIVE_MODEL_ID, output=H5_PATH, quiet=False)
    model = build_flood_model('sigmoid')     # matches the 2020 legacy head exactly
    model.load_weights(H5_PATH, by_name=True, skip_mismatch=True)
    # hard verification that the head weights really transferred:
    import h5py
    with h5py.File(H5_PATH, 'r') as hf:
        ref_kernel = hf['model_weights/dense/dense/kernel:0'][()]
    if not np.allclose(ref_kernel, model.get_layer('dense').get_weights()[0]):
        raise RuntimeError('weight transfer verification failed')
    LEGACY_SIGMOID = True
    model_source = 'pre-trained (Google Drive)'
    print('✅ Pre-trained flood model ready!  (source:', model_source + ')')
except Exception as e:
    print('⚠️  Pre-trained model unavailable →', str(e)[:120])
    print('🔁  Training a fresh model automatically (takes a few minutes)...\n')
    model, model_source = train_fresh_model(epochs=10)
    LEGACY_SIGMOID = False

print('\nModel in use :', model_source)
if LEGACY_SIGMOID:
    print('Note         : legacy model outputs 2 independent sigmoid scores;')
    print('               probabilities shown to you are normalised (sum to 1).')

## 3️⃣ Upload Your Images

Choose one of three ways when prompted:

| Option | Best for | How |
|--------|----------|-----|
| **1 — Multi-select** | a few images | pick one or many `jpg / jpeg / png / webp / bmp` files in the dialog |
| **2 — ZIP batch** | large batches | upload one `.zip`; every image inside is extracted automatically |
| **3 — Demo** | quick test | uses 12 test images from the flood dataset (auto-downloaded, ~52 MB) |

In [ ]:
# ============================================================================
# 3. UPLOAD — multi-select, zip batch, or demo images
# ============================================================================
from google.colab import files

UPLOAD_DIR = 'uploaded_images'
shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}


def _save_upload(payload: dict):
    saved = []
    for name, data in payload.items():
        path = os.path.join(UPLOAD_DIR, os.path.basename(name))
        with open(path, 'wb') as fh:
            fh.write(data)
        saved.append(path)
    return saved


def _scan_images(folder):
    found = []
    for root, _, fnames in os.walk(folder):
        for f in fnames:
            if os.path.splitext(f)[1].lower() in IMG_EXT:
                found.append(os.path.join(root, f))
    return sorted(found)


mode = input('Type 1, 2 or 3 + Enter  [1 = pick images | 2 = upload .zip | 3 = demo]: ').strip() or '1'
image_paths = []

if mode == '1':
    print('👆 Select one or more images in the dialog...')
    image_paths = [p for p in _save_upload(files.upload())
                   if os.path.splitext(p)[1].lower() in IMG_EXT]

elif mode == '2':
    print('👆 Select a .zip archive in the dialog...')
    for zp in _save_upload(files.upload()):
        if zp.lower().endswith('.zip'):
            dest = zp[:-4] + '_extracted'
            with zipfile.ZipFile(zp) as z:
                z.extractall(dest)
            image_paths += _scan_images(dest)
        elif os.path.splitext(zp)[1].lower() in IMG_EXT:
            image_paths.append(zp)

elif mode == '3':
    print('🎬 Demo mode — downloading dataset and using its test images...')
    test_dir = get_dataset_dirs()['test']
    demo = _scan_images(test_dir)
    for i, src in enumerate(demo[:12], 1):
        dst = os.path.join(UPLOAD_DIR, f'demo_{i:02d}_{os.path.basename(src)}')
        shutil.copy(src, dst)
        image_paths.append(dst)

else:
    print('❌ Unknown option — run this cell again and type 1, 2 or 3.')

# validate: skip unreadable / corrupt files so nothing crashes later
valid_paths = []
for p in image_paths:
    try:
        with Image.open(p) as im:
            im.verify()
        valid_paths.append(p)
    except Exception:
        print('⚠️  Skipping unreadable file:', p)
image_paths = valid_paths

print(f'\n✅ {len(image_paths)} image(s) ready for detection')
if not image_paths:
    print('   Run this cell again and choose option 1, 2 or 3.')

## 4️⃣ Run Detection 🤖

Every image is resized to 224×224, normalised the way MobileNet expects, and classified.
Results are collected in a table and saved to `results.csv`.

In [ ]:
# ============================================================================
# 4. DETECTION — batch prediction over all uploaded images
# ============================================================================
def load_and_preprocess(path):
    with Image.open(path) as im:
        im = im.convert('RGB').resize(IMG_SIZE)
        arr = np.asarray(im, dtype=np.float32)
    return tf.keras.applications.mobilenet.preprocess_input(arr)


def detect(model, paths, batch_size=32, legacy_sigmoid=LEGACY_SIGMOID):
    '''Returns a DataFrame with verdict + probabilities for every image.'''
    rows = []
    proc = [load_and_preprocess(p) for p in paths]
    for start in range(0, len(proc), batch_size):
        chunk = np.stack(proc[start:start + batch_size])
        preds = model.predict(chunk, verbose=0)
        for p, path in zip(preds, paths[start:start + batch_size]):
            flood_raw, noflood_raw = float(p[0]), float(p[1])
            if legacy_sigmoid:                       # normalise legacy sigmoid scores
                total = flood_raw + noflood_raw
                flood_p = flood_raw / total if total > 0 else 0.5
            else:                                    # softmax head -> already valid probs
                flood_p = flood_raw
            verdict = LABELS[0] if flood_p >= 0.5 else LABELS[1]
            rows.append({'file': os.path.basename(path),
                         'path': path,
                         'verdict': verdict,
                         'flood_probability': round(flood_p, 4),
                         'no_flood_probability': round(1.0 - flood_p, 4),
                         'confidence': round(max(flood_p, 1 - flood_p), 4),
                         'flood_score_raw': round(flood_raw, 4),
                         'no_flood_score_raw': round(noflood_raw, 4)})
    return pd.DataFrame(rows)


if image_paths:
    print(f'🔎 Running detection on {len(image_paths)} image(s)...\n')
    results = detect(model, image_paths)
    results.to_csv('results.csv', index=False)
    print(f'✅ Done — saved results.csv')
    print(f'   Flooding  : {(results.verdict == "Flooding").sum()} image(s)')
    print(f'   No Flooding: {(results.verdict == "No Flooding").sum()} image(s)')
    display(results.drop(columns='path'))
else:
    print('ℹ️  No images uploaded — run Section 3 first.')

## 5️⃣ Visual Results — Annotated Gallery

Every image with its verdict. 🔴 red title = **Flooding**, 🟢 green = **No Flooding**.

In [ ]:
# ============================================================================
# 5. GALLERY — annotated image grid
# ============================================================================
if 'results' in globals() and len(results):
    show = results.head(24)                       # first 24 images (increase if you like)
    n = len(show)
    cols = 4
    rows_n = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4 * cols, 4.4 * rows_n),
                             constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis('off')
    for ax, (_, r) in zip(axes, show.iterrows()):
        img = Image.open(r.path).convert('RGB')
        ax.imshow(img)
        color = '#d62728' if r.verdict == 'Flooding' else '#2ca02c'
        ax.set_title(f"{r.file}\n{r.verdict} — {r.confidence*100:.1f}%",
                     fontsize=11, color=color)
    fig.suptitle('Flood Detection Results', fontsize=15)
    plt.show()
    if len(results) > 24:
        print(f'(showing first 24 of {len(results)} — see Section 6/7 for everything)')
else:
    print('ℹ️  Run Sections 3-4 first to upload images and detect.')

## 6️⃣ Statistics, Graphs & Queries 📊

A full dashboard: verdict distribution, confidence histogram, per-image probability ranking
and a top-N "most flooded" chart — plus ready-made queries you can tweak.

In [ ]:
# ============================================================================
# 6. DASHBOARD — distribution, histogram, ranking, borderline images
# ============================================================================
def build_dashboard(df: pd.DataFrame, save_path=None):
    fig = plt.figure(figsize=(14, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)
    n_flood = int((df.verdict == 'Flooding').sum())
    n_dry   = len(df) - n_flood

    # (1) verdict pie
    ax1 = fig.add_subplot(gs[0, 0])
    if n_flood and n_dry:
        ax1.pie([n_flood, n_dry], labels=['Flooding', 'No Flooding'],
                colors=['#d62728', '#2ca02c'], autopct='%1.1f%%',
                explode=(0.03, 0.03), startangle=90)
    else:
        ax1.pie([max(n_flood, 1)], labels=['Flooding' if n_flood else 'No Flooding'],
                colors=['#d62728' if n_flood else '#2ca02c'], autopct='%1.0f%%')
    ax1.set_title('Verdict distribution')

    # (2) confidence histogram
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(df.flood_probability, bins=20, color='#1f77b4', edgecolor='white')
    ax2.axvline(0.5, color='#d62728', linestyle='--', label='decision threshold (0.5)')
    ax2.set_title('Flood probability histogram'); ax2.set_xlabel('P(flood)')
    ax2.set_ylabel('images'); ax2.legend(); ax2.grid(alpha=.3)

    # (3) per-image ranking
    ax3 = fig.add_subplot(gs[1, 0])
    order = df.sort_values('flood_probability')
    colors = ['#d62728' if v == 'Flooding' else '#2ca02c' for v in order.verdict]
    ax3.barh(range(len(order)), order.flood_probability, color=colors)
    ax3.set_yticks(range(len(order)), [f[:18] for f in order.file], fontsize=7)
    ax3.axvline(0.5, color='black', linestyle='--', linewidth=0.8)
    ax3.set_title('Per-image flood probability'); ax3.set_xlabel('P(flood)')

    # (4) top-10 most flooded
    ax4 = fig.add_subplot(gs[1, 1])
    top = df.nlargest(min(10, len(df)), 'flood_probability').iloc[::-1]
    ax4.barh([f[:18] for f in top.file], top.flood_probability, color='#d62728')
    for i, (p, c) in enumerate(zip(top.flood_probability, top.confidence)):
        ax4.text(p, i, f' {p*100:.1f}%', va='center', fontsize=8)
    ax4.set_title('Top-10 most flooded images'); ax4.set_xlabel('P(flood)')

    if save_path:
        fig.savefig(save_path, dpi=110)
    plt.show()


if 'results' in globals() and len(results):
    build_dashboard(results, save_path='analysis_report.png')
    print('💾 Dashboard saved to analysis_report.png\n')

    print('═' * 62)
    print('QUERY RESULTS')
    print('═' * 62)
    nf = results[results.verdict == 'Flooding'].sort_values('confidence', ascending=False)
    nd = results[results.verdict == 'No Flooding'].sort_values('confidence', ascending=False)
    border = results[results.confidence < 0.60].sort_values('confidence')
    if len(nf):
        print(f'\n🔴 Most confident FLOOD    : {nf.iloc[0].file}  ({nf.iloc[0].flood_probability*100:.1f}%)')
    else:
        print('\n🔴 Most confident FLOOD    : none')
    if len(nd):
        print(f'🟢 Most confident NO FLOOD : {nd.iloc[0].file}  ({nd.iloc[0].no_flood_probability*100:.1f}%)')
    else:
        print('🟢 Most confident NO FLOOD : none')
    print(f'🟡 Borderline (<60% conf.) : {len(border)} image(s)')
    if len(border):
        print(border[['file', 'verdict', 'flood_probability']].to_string(index=False))
    print(f'\nMean P(flood) = {results.flood_probability.mean():.3f}   '
          f'Median = {results.flood_probability.median():.3f}   '
          f'Std = {results.flood_probability.std():.3f}')
else:
    print('ℹ️  Run Sections 3-4 first to upload images and detect.')

In [ ]:
# ============================================================================
# 6b. YOUR OWN QUERIES — edit and re-run as often as you like
# ============================================================================
if 'results' in globals() and len(results):
    # e.g. every image that is at least 70% likely to be flooded:
    threshold = 0.70        # <-- change me!
    hits = results[results.flood_probability >= threshold]
    print(f'Images with P(flood) >= {threshold:.0%}: {len(hits)}')
    display(hits[['file', 'verdict', 'flood_probability']])

    # quick summary stats
    display(results[['flood_probability', 'confidence']].describe().round(3))
else:
    print('ℹ️  Run Sections 3-4 first to upload images and detect.')

## 7️⃣ Export Everything & Download 📦

Builds a single **`flood_analysis_results.zip`** containing:

```
flood_analysis_results/
├── results.csv                     ← full table (probabilities, verdicts)
├── analysis_report.png             ← the 4-panel dashboard
├── analysis_summary.txt            ← human-readable run summary
├── labeled_copies/
│   ├── FLOOD_1.jpg  ...            ← images renamed by verdict
│   └── NO_FLOOD_1.jpg ...
└── model/                          ← the model used (H5 + Keras + TFLite)
```

The zip downloads automatically to your computer.

In [ ]:
# ============================================================================
# 7. EXPORT — csv + labeled copies + report + model, zipped & downloaded
# ============================================================================
if 'results' in globals() and len(results):
    EXPORT = 'flood_analysis_results'
    shutil.rmtree(EXPORT, ignore_errors=True)
    os.makedirs(f'{EXPORT}/labeled_copies', exist_ok=True)
    os.makedirs(f'{EXPORT}/model', exist_ok=True)

    # 1. csv + dashboard figure
    results.to_csv(f'{EXPORT}/results.csv', index=False)
    if os.path.exists('analysis_report.png'):
        shutil.copy('analysis_report.png', EXPORT)

    # 2. labeled copies: FLOOD_1.jpg / NO_FLOOD_1.jpg (sorted by confidence)
    flood_i = dry_i = 0
    for _, r in results.sort_values('flood_probability', ascending=False).iterrows():
        ext = os.path.splitext(r.file)[1].lower() or '.jpg'
        if r.verdict == 'Flooding':
            flood_i += 1; new = f'FLOOD_{flood_i}{ext}'
        else:
            dry_i += 1;   new = f'NO_FLOOD_{dry_i}{ext}'
        shutil.copy(r.path, os.path.join(f'{EXPORT}/labeled_copies', new))

    # 3. text summary
    now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(f'{EXPORT}/analysis_summary.txt', 'w') as fh:
        fh.write(textwrap.dedent(f'''\
        FLOOD DETECTION ANALYSIS SUMMARY
        ================================
        Run date      : {now}
        Model source  : {model_source}
        Images tested : {len(results)}
        Flooding      : {int((results.verdict == 'Flooding').sum())}
        No Flooding   : {int((results.verdict == 'No Flooding').sum())}
        Mean P(flood) : {results.flood_probability.mean():.3f}
        Median P(flood): {results.flood_probability.median():.3f}

        Note: the pre-trained model was trained mostly on street-level photos.
        Treat satellite / drone results as estimates.
        '''))

    # 4. the model itself
    if os.path.exists(H5_PATH):
        shutil.copy(H5_PATH, f'{EXPORT}/model/flood_model.h5')
    if os.path.exists(os.path.join(MODELS_DIR, 'flood_model_trained.keras')):
        shutil.copy(os.path.join(MODELS_DIR, 'flood_model_trained.keras'),
                    f'{EXPORT}/model/flood_model_trained.keras')

    # 5. zip + auto-download
    zip_name = shutil.make_archive('flood_analysis_results', 'zip', EXPORT)
    size_mb = os.path.getsize(zip_name) / 1e6
    print(f'📦 {zip_name} ({size_mb:.1f} MB) — starting download...')
    files.download(zip_name)
else:
    print('ℹ️  Run Sections 3-4 first to upload images and detect.')

## 8️⃣ Retrain the Model — Full Pipeline *(optional)*

Trains a **fresh** flood detector from scratch on the flood dataset (321 train / 81 valid /
72 test images) with early stopping, then shows training curves, the confusion matrix and a
per-class report. Also exports a **TFLite** version for mobile/edge apps.

After training, the new model **replaces the current one** — simply re-run Sections 4-7 to
re-analyse your uploaded images with it. (Expected test accuracy: ~95-98%, a few minutes on GPU.)

In [ ]:
# ============================================================================
# 8. RETRAIN — full pipeline + curves + confusion matrix + TFLite
# ============================================================================
retrained_model, retrain_hist = train_fresh_model(epochs=15)

# per-class confusion matrix for the fresh model
dirs = get_dataset_dirs()
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    dirs['test'], labels='inferred', label_mode='categorical',
    class_names=['flooding', 'normal'], image_size=IMG_SIZE,
    batch_size=BATCH, shuffle=False)
y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in test_ds_raw])
y_pred = np.argmax(retrained_model.predict(test_ds_raw.map(
    lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y)),
    verbose=0), 1)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(cm, cmap=plt.cm.Blues)
fig.colorbar(im, ax=ax)
ax.set_xticks([0, 1], LABELS, rotation=45); ax.set_yticks([0, 1], LABELS)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Retrained model — Confusion matrix')
plt.show()
print(classification_report(y_true, y_pred, target_names=LABELS, digits=4))

# switch the notebook over to the fresh model
model = retrained_model
model_source = 'freshly retrained (this session)'
LEGACY_SIGMOID = False
print('✅ model is now the retrained one — re-run Sections 4-7 to re-analyse your images.')

In [ ]:
# ============================================================================
# 8b. TFLite export — for Android / Raspberry Pi / microcontrollers
# ============================================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
TFLITE_PATH = os.path.join(MODELS_DIR, 'flood_model.tflite')
with open(TFLITE_PATH, 'wb') as fh:
    fh.write(tflite_model)
print(f'✅ Saved {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH)/1e6:.1f} MB) — '
      f'ideal for mobile & edge devices.')

## 9️⃣ Satellite-Image Fine-Tuning *(optional — improves satellite accuracy)*

The pre-trained model learned mostly from **street-level** photos. To make it accurate on
**satellite / drone / aerial** imagery, fine-tune it on top-down flood data.

**Your dataset options**

| Dataset | Type | Link |
|---|---|---|
| **FloodNet** | UAV/drone flood imagery (post-Hurricane Harvey), classification & segmentation | [github.com/BinaLab/FloodNet](https://github.com/BinaLab/FloodNet) |
| **Sen1Floods11** | Sentinel-1 radar + Sentinel-2 optical flood scenes, world-wide | [github.com/cloudtostreet/Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11) |
| **Your own zip** | any two-folder image dataset | upload below |

**Zip format required** — one folder per class, images inside:

```
my_satellite_data.zip
├── flooded/            (or water/, flood/, inundated/ ...)
│   ├── a1.jpg ...
└── clear/              (or dry/, no_water/, normal/ ...)
    ├── b1.jpg ...
```

The cell below uploads your zip, auto-detects the two class folders, fine-tunes the current
model at a low learning rate (so it keeps its street-level knowledge) and reports metrics.
> 💡 Works best with **100+ images per class**. A few minutes on GPU.

In [ ]:
# ============================================================================
# 9. SATELLITE FINE-TUNING — upload a 2-folder dataset zip and adapt the model
# ============================================================================
from google.colab import files

SAT_ROOT = 'satellite_dataset'
choice = input('Type 1 + Enter to upload your satellite dataset .zip (or just Enter to skip): ').strip()

if choice != '1':
    print('ℹ️  Skipped. Re-run this cell any time to adapt the model to satellite imagery.')
else:
    shutil.rmtree(SAT_ROOT, ignore_errors=True)
    os.makedirs(SAT_ROOT, exist_ok=True)
    print('👆 Select your dataset .zip (two class folders inside)...')
    for name, data in files.upload().items():
        zpath = os.path.join(SAT_ROOT, os.path.basename(name))
        with open(zpath, 'wb') as fh:
            fh.write(data)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(zpath) as z:
                z.extractall(SAT_ROOT)
    print('✅ extracted to', SAT_ROOT)

    # --- auto-detect the two class folders -------------------------------
    IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    class_dirs = []
    for root, dirs, fnames in os.walk(SAT_ROOT):
        if root == SAT_ROOT:
            continue
        if '__macosx' in root.lower():
            continue
        n_img = sum(1 for f in fnames if os.path.splitext(f)[1].lower() in IMG_EXT)
        if n_img >= 8:                                   # needs a usable number of images
            class_dirs.append(root)
    class_dirs = sorted(class_dirs)

    if len(class_dirs) != 2:
        raise RuntimeError(
            f'Expected exactly 2 class folders with images, found {len(class_dirs)}: '
            f'{class_dirs}. Please organise the zip as two folders, one per class.')

    sat_classes = [os.path.basename(c) for c in class_dirs]
    # decide which folder means "Flooding" (heuristic on folder names)
    FLOOD_HINTS = ('flood', 'water', 'inund', 'wet')
    first = sat_classes[0].lower()
    flood_idx = 0 if any(h in first for h in FLOOD_HINTS) else 1
    SAT_LABELS = ['Flooding', 'No Flooding'] if flood_idx == 0 else ['No Flooding', 'Flooding']
    print(f'📁 Class folders detected:')
    for i, c in enumerate(sat_classes):
        print(f'   class {i} -> {c!r}  =>  {SAT_LABELS[i]}')
    print('   (rename folders to include flood/water or dry/normal to control this mapping)')

    # --- datasets (auto 80/20 split) --------------------------------------
    common = dict(labels='inferred', label_mode='categorical',
                  class_names=sat_classes, image_size=IMG_SIZE, batch_size=BATCH,
                  seed=SEED, validation_split=0.2)
    sat_train = tf.keras.utils.image_dataset_from_directory(SAT_ROOT, subset='training', **common)
    sat_val   = tf.keras.utils.image_dataset_from_directory(SAT_ROOT, subset='validation', **common)
    sat_train = sat_train.map(
        lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    sat_val = sat_val.map(
        lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    # --- fine-tune FROM the current model (low LR keeps prior knowledge) ---
    sat_model = build_flood_model('softmax')
    sat_model.set_weights(model.get_weights())           # start from current best weights
    for layer in sat_model.layers[:-40]:                 # unfreeze a bit more for adaptation
        layer.trainable = False
    sat_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                      loss='categorical_crossentropy', metrics=['accuracy'])

    print(f'\n🛰️  Fine-tuning on satellite data '
          f'({sum(1 for l in sat_model.layers if l.trainable)}/{len(sat_model.layers)} layers trainable)...')
    sat_hist = sat_model.fit(sat_train, validation_data=sat_val, epochs=8, verbose=2,
                             callbacks=[tf.keras.callbacks.EarlyStopping(
                                 monitor='val_accuracy', patience=2,
                                 restore_best_weights=True)])

    # curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(sat_hist.history['accuracy'], 'o-', label='train')
    axes[0].plot(sat_hist.history['val_accuracy'], 's-', label='validation')
    axes[0].set_title('Satellite fine-tune - Accuracy'); axes[0].set_xlabel('epoch')
    axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(sat_hist.history['loss'], 'o-', label='train')
    axes[1].plot(sat_hist.history['val_loss'], 's-', label='validation')
    axes[1].set_title('Satellite fine-tune - Loss'); axes[1].set_xlabel('epoch')
    axes[1].legend(); axes[1].grid(alpha=.3)
    plt.show()

    # confusion matrix on the validation split
    y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in sat_val])
    y_pred = np.argmax(sat_model.predict(sat_val, verbose=0), 1)
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    im = ax.imshow(cm, cmap=plt.cm.Blues); fig.colorbar(im, ax=ax)
    ax.set_xticks([0, 1], SAT_LABELS, rotation=45); ax.set_yticks([0, 1], SAT_LABELS)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Satellite model — Confusion matrix (validation)')
    plt.show()
    print(classification_report(y_true, y_pred, target_names=SAT_LABELS, digits=4))

    # save + switch the notebook over to the satellite model
    sat_model.save(os.path.join(MODELS_DIR, 'flood_model_satellite.keras'))
    model = sat_model
    model_source = 'satellite fine-tuned (this session)'
    LEGACY_SIGMOID = False
    print(f'✅ Saved {MODELS_DIR}/flood_model_satellite.keras')
    print('✅ model is now the satellite-tuned one — re-run Sections 4-7 to re-analyse your images.')

---
## 📎 Summary & Credits

**What this notebook did**
1. Set itself up and picked the best available hardware (GPU when possible)
2. Loaded the pre-trained flood detector (or trained a fresh one automatically)
3. Let you upload satellite / drone / street-level images (single, many, or a whole zip)
4. Produced verdicts + probabilities for every image
5. Showed a gallery, a full statistics dashboard and ready-made queries
6. Exported **everything** (csv, dashboard, labeled copies, summary, model) as one zip
7. Gave you an optional one-click retraining pipeline + TFLite export
8. Offered optional satellite-image fine-tuning for higher satellite accuracy

**Credits & sources**
- Code inspired by [deeplizard](https://deeplizard.com/)
- Flooding images: authors of [*Detecting floodwater on roadways from image data with
  handcrafted features and deep transfer learning*](https://arxiv.org/pdf/1909.00125.pdf)
- Base model: [MobileNet](https://arxiv.org/abs/1704.04861) (ImageNet weights), TensorFlow / Keras

> ⚠️ **Disclaimer** — this is a research demo, not a life-safety system. Always confirm flood
> conditions through official channels.